<a href="https://colab.research.google.com/github/kieu-collab/CreatorRevenuePrediction/blob/main/baitapcanhan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# 1. Cài đặt môi trường và lấy source code từ GitHub
import os, sys, subprocess
REPO_URL = "https://github.com/kieu-collab/CreatorRevenuePrediction.git"
PROJECT_DIR = "/content/CreatorRevenuePrediction"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], check=False)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Project directory:", os.getcwd())

Project directory: /content/CreatorRevenuePrediction


In [9]:
# 2. Đọc, làm sạch dữ liệu và tạo marketing features
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.data_processing import load_and_clean, validate_dataset
from src.feature_engineering import build_features

DATA_PATH = Path("data/creator_campaign.csv")
df = load_and_clean(DATA_PATH, require_target=True)
warnings = validate_dataset(df, require_target=True)
features = build_features(df)

print(f"Rows: {len(df):,} | Creators: {df['creator_id'].nunique():,}")
print("Validation warnings:", warnings)
display(df.head())
display(features[["creator_power_score", "planned_reach", "campaign_intensity"]].describe().T)

Rows: 4,798 | Creators: 3,132
Validation warnings: []


,creator_id,creator_name,brand_name,creator_niche,product_category,campaign_date,followers,avg_views,engagement_rate,product_price,...,planned_live_sessions,products_promoted,historical_conversion_rate,brand_fit_score,creator_cost,gross_margin_rate,orders,gmv,revenue,source_file
0,senabeauty,senabeauty,Lubylab Official Mall,Beauty & Personal Care,Beauty & Personal Care,2025-10-08,611300.0,91658.827338,0.04,1710000.0,...,139.0,191.0,0.02,70.0,15282500.0,0.35,36248.0,6.198463e+10,6.198463e+10,creator_campaign.csv
1,phuongsena,phuongsena,Lubylab Official Mall,Beauty & Personal Care,Beauty & Personal Care,2025-10-08,160500.0,37840.768293,0.04,1710000.0,...,158.0,186.0,0.02,70.0,4012500.0,0.35,27368.0,4.679939e+10,4.679939e+10,creator_campaign.csv
2,quyenleo,quyenleo,Lubylab Official Mall,Beauty & Personal Care,Beauty & Personal Care,2025-10-08,4700000.0,411998.800000,0.04,1710000.0,...,5.0,141.0,0.02,70.0,100000000.0,0.35,14167.0,2.422594e+10,2.422594e+10,creator_campaign.csv
3,bonphuongsena,bonphuongsena,Lubylab Official Mall,Beauty & Personal Care,Beauty & Personal Care,2025-10-08,52400.0,6786.662921,0.04,1710000.0,...,77.0,133.0,0.02,70.0,1310000.0,0.35,4464.0,7.634328e+09,7.634328e+09,creator_campaign.csv
4,giadinhsena,giadinhsena,Lubylab Official Mall,Beauty & Personal Care,Beauty & Personal Care,2025-10-08,46100.0,6732.728395,0.04,1710000.0,...,76.0,108.0,0.02,70.0,1152500.0,0.35,2477.0,4.237098e+09,4.237098e+09,creator_campaign.csv


,count,mean,std,min,25%,50%,75%,max
creator_power_score,4798.0,41.656904,9.110105e+00,12.816881,35.671301,42.015006,48.728330,6.333773e+01
planned_reach,4798.0,348794.570743,1.907981e+06,0.000000,0.000000,6836.200000,155581.500000,7.577109e+07
campaign_intensity,4798.0,1.480082,3.309190e+00,0.000000,0.066667,0.333333,1.633333,6.616667e+01


In [10]:
# 3. Xem kết quả benchmark model và các yếu tố ảnh hưởng doanh thu
metrics = pd.read_csv("models/metrics.csv")
importance = pd.read_csv("models/feature_importance.csv")
shap_summary = pd.read_csv("models/shap_summary.csv")

display(metrics.style.format({"MAE":"{:,.0f}", "RMSE":"{:,.0f}", "MAPE":"{:.2f}", "R2":"{:.3f}"}))
display(importance.head(15))
display(shap_summary.head(15))

,Model,MAE,RMSE,MAPE,R2
0,LightGBM,"211,674,625","2,255,013,239",112.19,0.448
1,XGBoost,"218,150,076","2,409,095,502",119.14,0.370
2,CatBoost,"221,121,492","2,552,040,360",116.30,0.293
3,Gradient Boosting,"225,398,914","2,599,122,864",111.57,0.266
4,Random Forest,"232,872,818","2,610,199,829",89.73,0.260
5,Neural Network,"1,399,863,453","31,714,159,352",143.23,-108.247
6,Linear Regression,"12,390,818,647","269,933,874,781",1382.08,-7913.413


,feature,importance_mean,importance_std
0,products_promoted,8.165994e+07,5.010953e+06
1,campaign_intensity,4.336200e+07,1.148681e+07
2,avg_views,3.432592e+07,7.285434e+06
3,planned_live_sessions,3.391671e+07,4.637789e+06
4,creator_power_score,3.138520e+07,7.999453e+06
5,brand_name,1.794029e+07,1.950126e+06
6,log_avg_views,1.690830e+07,5.466730e+06
7,planned_reach,1.433761e+07,1.991507e+06
8,log_followers,7.376401e+06,3.504831e+06
9,product_price,6.183394e+06,3.506734e+06


,feature,mean_abs_shap_log_revenue
0,products_promoted,0.780424
1,avg_views,0.644455
2,campaign_intensity,0.425827
3,creator_power_score,0.413400
4,planned_reach,0.411105
5,product_price,0.343396
6,log_avg_views,0.270394
7,followers,0.167974
8,log_followers,0.155023
9,engaged_views,0.143219


In [11]:
# 4. Train lại các mô hình và tự động lưu model tốt nhất
# Có thể đổi tập selected_models nếu muốn benchmark nhanh hơn.
from src.train_model import train_and_save

selected_models = {
    "Linear Regression",
    "Random Forest",
    "Gradient Boosting",
    "XGBoost",
    "LightGBM",
    "CatBoost",
    "Neural Network",
}
metadata = train_and_save(DATA_PATH, Path("models"), selected_models=selected_models)
print("Best model:", metadata["best_model"])
display(pd.read_csv("models/metrics.csv"))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Best model: LightGBM


,Model,MAE,RMSE,MAPE,R2
0,LightGBM,2.116746e+08,2.255013e+09,112.186436,0.447666
1,XGBoost,2.181501e+08,2.409096e+09,119.141249,0.369606
2,CatBoost,2.211215e+08,2.552040e+09,116.303570,0.292577
3,Gradient Boosting,2.253989e+08,2.599123e+09,111.566869,0.266234
4,Random Forest,2.328728e+08,2.610200e+09,89.726203,0.259966
5,Neural Network,1.399863e+09,3.171416e+10,143.234301,-108.247211
6,Linear Regression,1.239082e+10,2.699339e+11,1382.076390,-7913.413331


In [12]:
# 5. Dự báo doanh thu cho từng KOL và tạo bảng xếp hạng
from src.batch_prediction import generate_oof_predictions

summary = generate_oof_predictions(
    data_path=DATA_PATH,
    model_path=Path("models/best_model.pkl"),
    output_dir=Path("outputs"),
    n_splits=5,
)
detail = pd.read_csv("outputs/kol_revenue_predictions.csv")
ranking = pd.read_csv("outputs/kol_revenue_ranking.csv")

print(summary)
display(ranking.head(20).style.format({
    "mean_predicted_revenue":"{:,.0f}",
    "median_predicted_revenue":"{:,.0f}",
    "mean_expected_orders":"{:,.0f}",
    "mean_expected_roi_pct":"{:.1f}",
}))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


{'generated_at_utc': '2026-09-06T13:45:41.975819+00:00', 'model': 'LightGBM', 'protocol': '5-fold GroupKFold by creator_id', 'rows': 4798, 'creators': 3132, 'metrics': {'MAE': 129965844.34548312, 'RMSE': 1259042117.0202336, 'MAPE': 118.85978429009707, 'R2': 0.4836764907762222}, 'interval_log_residual_q10': -1.3376884948034418, 'interval_log_residual_q90': 1.4437792246219217, 'outputs': ['kol_revenue_predictions.csv', 'kol_revenue_ranking.csv'], 'warning': 'Current seed data contains same-period activity proxies and default marketing fields; use rankings as MVP evidence only.'}


,revenue_rank,creator_id,creator_name,campaigns_observed,brands_observed,followers,avg_views,creator_power_score,mean_predicted_revenue,median_predicted_revenue,mean_actual_revenue,total_actual_revenue,mean_expected_orders,mean_expected_roi_pct,median_absolute_percentage_error
0,1,focallure.vn,focallure.vn,1,1,592300.000000,56287.285714,51.641824,"15,962,177,328","15,962,177,328",4274825969.640000,4274825969.640000,"9,793",37629.3,273.399466
1,2,cerave.vietnam_official,cerave.vietnam_official,1,1,456900.000000,123628.645631,52.703699,"10,779,481,517","10,779,481,517",7548592408.620000,7548592408.620000,"56,743",32929.7,42.801213
2,3,3ce_vietnam,3ce_vietnam,1,1,433300.000000,177238.293210,53.291887,"10,622,622,150","10,622,622,150",9260949938.389999,9260949938.389999,"122,536",34221.9,14.703375
3,4,lorealparis_vn,lorealparis_vn,1,1,615200.000000,59541.295423,51.805265,"9,673,323,321","9,673,323,321",12555198152.389999,12555198152.389999,"35,340",21913.4,22.953639
4,5,cocoon.vietnam,cocoon.vietnam,1,1,623900.000000,208827.024510,54.162603,"9,054,490,522","9,054,490,522",65036659837.190002,65036659837.190002,"40,386",20217.8,86.077867
5,6,quynhanh_23291702,quynhanh_23291702,1,1,859700.000000,72349.482759,52.686942,"8,914,991,137","8,914,991,137",8509390630.440000,8509390630.440000,"41,005",14417.8,4.766505
6,7,colorkey_luminous.vn,colorkey_luminous.vn,1,1,218200.000000,32767.701987,49.085980,"7,311,091,534","7,311,091,534",7327877135.190000,7327877135.190000,"73,375",46808.9,0.229065
7,8,olayvietnam,olayvietnam,1,1,297600.000000,476768.848214,54.550970,"7,275,323,671","7,275,323,671",8204124653.540000,8204124653.540000,"20,584",34125.3,11.321147
8,9,merzy_vn,merzy_vn,1,1,149800.000000,63210.625000,49.725492,"7,018,384,550","7,018,384,550",3027637411.820000,3027637411.820000,"37,427",65492.4,131.810603
9,10,florasis.vn,florasis.vn,1,1,72300.000000,51313.079137,48.207481,"6,831,538,759","6,831,538,759",3350485192.790000,3350485192.790000,"2,340",132184.3,103.896999
